In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical/data_clinical_patient.txt


In [2]:
import os

print("="*80)
print("KAGGLE INPUT DATASETS")
print("="*80)

for dirname, dirnames, filenames in os.walk("/kaggle/input"):
    print(f"\n📂 {dirname}")
    for f in filenames:
        print(f"   └── {f}")

KAGGLE INPUT DATASETS

📂 /kaggle/input

📂 /kaggle/input/datasets

📂 /kaggle/input/datasets/anishapanja

📂 /kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical
   └── data_clinical_patient.txt


In [3]:
import os

print("Attached Kaggle Datasets:\n")

for item in os.listdir("/kaggle/input"):
    print(item)

Attached Kaggle Datasets:

datasets


In [4]:
import os

for dirname, dirnames, filenames in os.walk("/kaggle/input/datasets"):
    print(f"\n📂 {dirname}")
    for f in filenames:
        print(f"   └── {f}")


📂 /kaggle/input/datasets

📂 /kaggle/input/datasets/anishapanja

📂 /kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical
   └── data_clinical_patient.txt


In [5]:
import requests
import json
import pandas as pd

CLINICAL_PATH = "/kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical/data_clinical_patient.txt"

# Load clinical file
df = pd.read_csv(
    CLINICAL_PATH,
    sep="\t",
    comment="#",
    low_memory=False
)

# Get C8 patients with valid subtype labels
c8 = df[
    df["PATIENT_ID"].str.startswith("TCGA-C8-") &
    df["SUBTYPE"].notna() &
    (df["SUBTYPE"] != "BRCA_Normal")
].copy()

print("Subtype Distribution:")
print(c8["SUBTYPE"].value_counts())

patient_ids = c8["PATIENT_ID"].tolist()

# Query GDC
filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": ["TCGA-BRCA"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": ["Slide Image"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "cases.submitter_id",
                "value": patient_ids
            }
        }
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields": "file_id,file_name,file_size,cases.submitter_id",
    "format": "JSON",
    "size": "500"
}

response = requests.get(
    "https://api.gdc.cancer.gov/files",
    params=params
)

response.raise_for_status()

hits = response.json()["data"]["hits"]

rows = []

for h in hits:

    pid = h["cases"][0]["submitter_id"]

    subtype = c8.loc[
        c8["PATIENT_ID"] == pid,
        "SUBTYPE"
    ].values[0]

    rows.append({
        "patient_id": pid,
        "subtype": subtype,
        "file_id": h["file_id"],
        "file_name": h["file_name"],
        "file_size_mb": round(h["file_size"] / (1024**2), 2)
    })

manifest = pd.DataFrame(rows)

manifest.to_csv(
    "/kaggle/working/c8_manifest.csv",
    index=False
)

print("\nManifest created successfully!")
print(manifest.head())

print(f"\nTotal slides: {len(manifest)}")
print(f"Patients: {manifest['patient_id'].nunique()}")

print("\nSaved to:")
print("/kaggle/working/c8_manifest.csv")

Subtype Distribution:
SUBTYPE
BRCA_Her2     15
BRCA_LumB     14
BRCA_LumA     12
BRCA_Basal     6
Name: count, dtype: int64

Manifest created successfully!
     patient_id     subtype                               file_id  \
0  TCGA-C8-A1HI   BRCA_LumA  1bf2c09e-854f-414f-9b5e-2ad8a5176abd   
1  TCGA-C8-A12Y   BRCA_LumA  f5c54700-cb25-4cb3-aea5-59a1625b694f   
2  TCGA-C8-A12Y   BRCA_LumA  685d0657-860d-4ed6-b434-b24976b09333   
3  TCGA-C8-A131  BRCA_Basal  d42eb4ce-d300-46be-8b46-059a98c2ddb8   
4  TCGA-C8-A1HK   BRCA_Her2  e7aa1c87-1bf2-4415-a4dc-b60279f7b35c   

                                           file_name  file_size_mb  
0  TCGA-C8-A1HI-01Z-00-DX1.C6D0F8B8-55ED-477F-BAF...        890.43  
1  TCGA-C8-A12Y-01A-01-TSA.12966473-7301-460a-b03...        132.07  
2  TCGA-C8-A12Y-01A-01-BSA.6b6ff967-048f-4197-a4e...        238.15  
3  TCGA-C8-A131-01Z-00-DX1.5CB27A29-9951-40B9-B4D...        829.86  
4  TCGA-C8-A1HK-01A-02-TSB.f5919eb9-e65f-4fbb-940...        387.09  

Total slides: 